# 05 - Walk-Forward Validation

This notebook demonstrates walk-forward cross-validation for volatility forecasting.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from pathlib import Path
from src.training.walk_forward import WalkForwardCV, create_walk_forward_splits
from src.evaluation.metrics import evaluate_forecast

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 1. Walk-Forward Configuration

In [ ]:
# Configuration
WF_CONFIG = {
    'refit_period': 66,          # Quarterly refit (~3 months)
    'train_window': 'expanding', # Expanding window
    'val_size': 252,             # 1 year validation
    'gap': 1,                    # Gap = forecast horizon
}

print("Walk-Forward Configuration:")
for k, v in WF_CONFIG.items():
    print(f"  {k}: {v}")

## 2. Create Sample Data

In [ ]:
# Create synthetic data spanning 2014-2026
np.random.seed(42)

dates = pd.date_range('2014-08-26', '2026-02-03', freq='B')
n = len(dates)

# Generate RV with realistic characteristics
rv = np.abs(np.random.randn(n) * 0.001 + 0.0005)

# Add some regime changes
# COVID period (March 2020)
covid_start = pd.Timestamp('2020-03-01')
covid_end = pd.Timestamp('2020-06-01')
covid_mask = (dates >= covid_start) & (dates <= covid_end)
rv[covid_mask] *= 3

# 2022 crisis
crisis_2022_start = pd.Timestamp('2022-02-24')
crisis_2022_end = pd.Timestamp('2022-06-01')
crisis_mask = (dates >= crisis_2022_start) & (dates <= crisis_2022_end)
rv[crisis_mask] *= 4

df = pd.DataFrame({
    'date': dates,
    'rv': rv,
    'rv_d': np.roll(rv, 1),
    'rv_w': pd.Series(rv).rolling(5).mean().values,
    'rv_m': pd.Series(rv).rolling(22).mean().values,
})

df = df.iloc[22:].reset_index(drop=True)
print(f"Data range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Total observations: {len(df)}")

In [ ]:
# Plot RV time series
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(df['date'], df['rv'], linewidth=0.8)
ax.axvline(pd.Timestamp('2020-01-01'), color='red', linestyle='--', label='WF Start')
ax.axvspan(covid_start, covid_end, alpha=0.3, color='red', label='COVID')
ax.axvspan(crisis_2022_start, crisis_2022_end, alpha=0.3, color='orange', label='2022 Crisis')

ax.set_xlabel('Date')
ax.set_ylabel('Realized Volatility')
ax.set_title('RV Time Series with Crisis Periods')
ax.legend()

plt.tight_layout()
plt.show()

## 3. Walk-Forward Splits Visualization

In [ ]:
# Create walk-forward splits
wf_cv = WalkForwardCV(
    refit_period=WF_CONFIG['refit_period'],
    train_window=WF_CONFIG['train_window'],
    val_size=WF_CONFIG['val_size'],
    gap=WF_CONFIG['gap'],
)

# Get splits for walk-forward period (2020-2026)
wf_start = pd.Timestamp('2020-01-01')
wf_df = df[df['date'] >= wf_start].reset_index(drop=True)

# Generate split indices
splits = list(wf_cv.split(wf_df['date']))
print(f"Number of refit periods: {len(splits)}")

In [ ]:
# Visualize first few splits
fig, ax = plt.subplots(figsize=(14, 6))

colors = plt.cm.tab10(np.linspace(0, 1, min(10, len(splits))))

for i, (train_idx, val_idx, test_idx) in enumerate(splits[:10]):
    y_pos = i
    
    # Train period
    train_start = wf_df.loc[train_idx[0], 'date']
    train_end = wf_df.loc[train_idx[-1], 'date']
    ax.barh(y_pos, (train_end - train_start).days, left=train_start,
           height=0.3, color='blue', alpha=0.7, label='Train' if i == 0 else '')
    
    # Test period
    test_start = wf_df.loc[test_idx[0], 'date']
    test_end = wf_df.loc[test_idx[-1], 'date']
    ax.barh(y_pos, (test_end - test_start).days, left=test_start,
           height=0.3, color='green', alpha=0.7, label='Test' if i == 0 else '')

ax.set_yticks(range(min(10, len(splits))))
ax.set_yticklabels([f'Split {i+1}' for i in range(min(10, len(splits)))])
ax.set_xlabel('Date')
ax.set_title('Walk-Forward Cross-Validation Splits')
ax.legend()

plt.tight_layout()
plt.show()

## 4. Simulated Walk-Forward Results

In [ ]:
# Simulate walk-forward predictions
all_predictions = []

for i, (train_idx, val_idx, test_idx) in enumerate(splits):
    # Get test data
    test_data = wf_df.loc[test_idx].copy()
    y_true = test_data['rv'].values
    
    # Simulated predictions (in practice, would train model here)
    y_pred_har = y_true + np.random.randn(len(y_true)) * 0.0002
    y_pred_lgbm = y_true + np.random.randn(len(y_true)) * 0.00015
    
    test_data['y_pred_har'] = np.abs(y_pred_har) + 1e-8
    test_data['y_pred_lgbm'] = np.abs(y_pred_lgbm) + 1e-8
    test_data['split_id'] = i
    
    all_predictions.append(test_data[['date', 'rv', 'y_pred_har', 'y_pred_lgbm', 'split_id']])

wf_predictions = pd.concat(all_predictions, ignore_index=True)
print(f"Walk-forward predictions: {len(wf_predictions)} observations")
wf_predictions.head()

## 5. Performance Over Time

In [ ]:
# Calculate rolling RMSE
wf_predictions['sq_err_har'] = (wf_predictions['rv'] - wf_predictions['y_pred_har']) ** 2
wf_predictions['sq_err_lgbm'] = (wf_predictions['rv'] - wf_predictions['y_pred_lgbm']) ** 2

rolling_rmse_har = wf_predictions.set_index('date')['sq_err_har'].rolling(66).mean().apply(np.sqrt)
rolling_rmse_lgbm = wf_predictions.set_index('date')['sq_err_lgbm'].rolling(66).mean().apply(np.sqrt)

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(rolling_rmse_har.index, rolling_rmse_har.values, label='HAR-RV', linewidth=1.5)
ax.plot(rolling_rmse_lgbm.index, rolling_rmse_lgbm.values, label='LightGBM', linewidth=1.5)

ax.axvspan(covid_start, covid_end, alpha=0.2, color='red', label='COVID')
ax.axvspan(crisis_2022_start, crisis_2022_end, alpha=0.2, color='orange', label='2022 Crisis')

ax.set_xlabel('Date')
ax.set_ylabel('Rolling RMSE (66-day)')
ax.set_title('Walk-Forward Performance Over Time')
ax.legend()

plt.tight_layout()
plt.show()

## 6. Performance by Year

In [ ]:
# Annual metrics
wf_predictions['year'] = wf_predictions['date'].dt.year

annual_metrics = []
for year in wf_predictions['year'].unique():
    year_data = wf_predictions[wf_predictions['year'] == year]
    
    for model in ['har', 'lgbm']:
        y_true = year_data['rv'].values
        y_pred = year_data[f'y_pred_{model}'].values
        
        metrics = evaluate_forecast(y_true, y_pred)
        
        annual_metrics.append({
            'Year': year,
            'Model': model.upper(),
            'QLIKE': metrics.qlike,
            'RMSE': metrics.rmse,
            'R²': metrics.r2,
            'N': len(year_data),
        })

annual_df = pd.DataFrame(annual_metrics)
annual_df.pivot(index='Year', columns='Model', values='QLIKE')

In [ ]:
# Plot annual QLIKE
fig, ax = plt.subplots(figsize=(12, 5))

years = annual_df['Year'].unique()
x = np.arange(len(years))
width = 0.35

har_qlike = annual_df[annual_df['Model'] == 'HAR']['QLIKE'].values
lgbm_qlike = annual_df[annual_df['Model'] == 'LGBM']['QLIKE'].values

ax.bar(x - width/2, har_qlike, width, label='HAR-RV')
ax.bar(x + width/2, lgbm_qlike, width, label='LightGBM')

ax.set_xlabel('Year')
ax.set_ylabel('QLIKE')
ax.set_title('Annual QLIKE by Model')
ax.set_xticks(x)
ax.set_xticklabels(years)
ax.legend()

plt.tight_layout()
plt.show()

## 7. Overall Walk-Forward Results

In [ ]:
# Overall metrics
overall_results = []

for model in ['har', 'lgbm']:
    y_true = wf_predictions['rv'].values
    y_pred = wf_predictions[f'y_pred_{model}'].values
    
    metrics = evaluate_forecast(y_true, y_pred)
    
    overall_results.append({
        'Model': model.upper(),
        'QLIKE': metrics.qlike,
        'MSE': metrics.mse,
        'RMSE': metrics.rmse,
        'MAE': metrics.mae,
        'R²': metrics.r2,
    })

overall_df = pd.DataFrame(overall_results)
print("\n" + "="*60)
print("WALK-FORWARD VALIDATION RESULTS (2020-2026)")
print("="*60)
print(f"\nTotal predictions: {len(wf_predictions)}")
print(f"Refit periods: {len(splits)}")
print(f"\nOverall Metrics:")
print(overall_df.to_string(index=False))

## 8. Save Walk-Forward Predictions

In [ ]:
# Save predictions
OUTPUT_DIR = Path("../data/predictions/walkforward_2020_2026")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# wf_predictions.to_parquet(OUTPUT_DIR / "predictions_h1.parquet", index=False)
# print(f"Saved to {OUTPUT_DIR / 'predictions_h1.parquet'}")

print(f"\nWalk-forward predictions shape: {wf_predictions.shape}")
print(f"Columns: {wf_predictions.columns.tolist()}")